# Notebook 05 — Extractor de BI-RADS declarado

## Qué hace este notebook

Implemento el primer módulo del MVP: un extractor que lee el `Full_Report` de un informe mamográfico y devuelve la categoría **BI-RADS declarada por el radiólogo en su CONCLUSIÓN**. Esto NO es el BI-RADS predicho por el modelo de IA (eso lo hace DistilBETO en el nb 04); es la categoría que el médico escribió textualmente.

## Por qué este módulo es importante

El proyecto valida si **lo que el radiólogo escribió como conclusión coincide con lo que desarrolló en el cuerpo del informe y con la recomendación que dio**. Para eso necesito extraer la categoría declarada con confiabilidad, manejando todas las variantes de redacción que aparecen en informes reales.

## Decisiones de diseño basadas en el corpus

Antes de codificar exploré el corpus (4 357 informes) y encontré:

- 100% de los informes tienen encabezado de conclusión (`CONCLUSIÓN:` 67.7%, `VALORACIÓN:` 32.3%)
- 99.95% de los informes tienen al menos una mención de BI-RADS detectable
- Solo 2 informes tienen typos que ocultan el BI-RADS (`BI-RADS O` con letra O, `BI-RADAS 4`)
- Solo 2 informes tienen menciones múltiples; en ambos casos la del bloque conclusión es la definitiva
- 0% del corpus usa romanos o palabras (cubro estos como soporte latente para portabilidad)

## Estructura del notebook

1. Imports y carga del corpus
2. Definición de patrones (encabezados de conclusión, BI-RADS, typos)
3. Funciones auxiliares (conversión de numeraciones, localización del bloque conclusión)
4. Función pública principal `extraer_birads()`
5. Tests inline con 12 casos de prueba
6. Validación sobre el corpus completo (4 357 informes)
7. Análisis de divergencias contra la etiqueta del dataset
8. Guardar resultados y conclusiones


---

## Paso 1 — Imports y carga del corpus

**Qué espero ver**: el DataFrame cargado con 4 357 filas y las columnas esperadas, incluida `Full_Report` (texto original sin normalizar).


In [1]:
import re
import unicodedata
import pandas as pd
from collections import Counter
from typing import Optional, List, Dict, Any

DATA_PATH = "../data/processed/reports_cleaned.csv"
df = pd.read_csv(DATA_PATH)

print(f"Filas: {len(df)}")
print(f"Columnas: {df.columns.tolist()}")

assert "Full_Report" in df.columns, "Falta Full_Report"
assert "BI-RADS" in df.columns, "Falta BI-RADS"
print("\nDataset cargado correctamente")


Filas: 4357
Columnas: ['Full_Report', 'Conclusion', 'Recommendations', 'BI-RADS', 'Full_Report_clean', 'Conclusion_clean', 'Recommendations_clean', 'len_report']

Dataset cargado correctamente


---

## Paso 2 — Definición de patrones

Defino tres bloques de patrones, ordenados del más estricto al más permisivo. Cada bloque se documenta con la decisión que hay detrás.

### 2.1 Encabezados de conclusión

Cubro todas las variantes que aparecen en el corpus + variantes habituales en otros centros (aportadas por el dominio clínico).


In [2]:
# Encabezados que marcan el inicio del bloque de CONCLUSION del informe
# Ordenados de mas especifico a mas general para resolver coincidencias ambiguas
ENCABEZADOS_CONCLUSION = [
    r"conclusi[oó]n\s+e?\s*impresi[oó]n\s+diagn[oó]stica",
    r"hallazgos\s+y\s+conclusi[oó]n",
    r"impresi[oó]n\s+diagn[oó]stica\s+y?\s*recomendaciones?",
    r"impresi[oó]n\s+diagn[oó]stica",
    r"impresi[oó]n\s+final",
    r"diagn[oó]stico\s+presuntivo",
    r"diagn[oó]stico\s+radiol[oó]gico",
    r"opini[oó]n\s+del\s+radi[oó]logo",
    r"conclusi[oó]n",
    r"valoraci[oó]n",
    r"impresi[oó]n",
    r"diagn[oó]stico",
]

# Patron compuesto: cualquiera de los encabezados, con dos puntos opcionales
PATRON_ENCABEZADO_CONCLUSION = re.compile(
    r"\b(" + "|".join(ENCABEZADOS_CONCLUSION) + r")\s*:?",
    re.IGNORECASE
)

# Encabezados que marcan el FIN del bloque de conclusion (inicio de recomendaciones)
PATRON_INICIO_RECOMENDACIONES = re.compile(
    r"\b(recomendaci[oó]n(?:es)?|indicaci[oó]n(?:es)?|sugerenci[ao]s?|conducta\s+a?\s*seguir|plan)\s*:?",
    re.IGNORECASE
)

print(f"Encabezados de conclusion cubiertos: {len(ENCABEZADOS_CONCLUSION)}")
print("Patrones compilados correctamente")


Encabezados de conclusion cubiertos: 12
Patrones compilados correctamente


### 2.2 Patrones BI-RADS

Construyo una regex que captura las 26 variantes vistas en el corpus + los formatos latentes (romanos, palabras). El patrón principal devuelve grupos con el número y la subcategoría por separado para procesarlos limpiamente.


In [3]:
# Patron BI-RADS principal
# Cubre: BI-RADS, BIRADS, BI RADS, BI*RADS, BI.RADS, Bi-Rads (mayusculas mezcladas)
# Con/sin simbolo (R) en distintas posiciones
# Con/sin dos puntos
# Captura: numero arabigo 0-6, romano I-VI, o palabra (cero, uno, dos...)
# Opcionalmente subcategoria a/b/c

PATRON_BIRADS_PRINCIPAL = re.compile(
    r'\bbi\s*[-*.]?\s*rad[s]?'           # BI-RADS, BIRADS, BI RADS, BI*RADS, BI.RADS, BI-RAD
    r'\s*®?\s*'                          # opcional simbolo (R)
    r':?\s*'                             # opcional dos puntos
    r'\(?\s*'                            # opcional parentesis
    r'('
    r'0?[0-6](?:\s*-\s*[0-6])?'          # arabigo: 0-6 con cero opcional a la izquierda (ej: BI-RADS01)
    r'|'
    r'VI|IV|V|III|II|I'                  # romanos
    r'|'
    r'cero|uno|dos|tres|cuatro|cinco|seis'  # palabras
    r')'
    r'\s*'
    r'([abc])?'                          # subcategoria opcional (4a, 4b, 4c)
    r'\s*\)?',
    re.IGNORECASE
)

# Patron TOLERANTE para typos (segunda pasada solo si el principal no encuentra)
# Maneja: letra O en lugar de 0, letra l en lugar de 1, BI-RADAS en lugar de BI-RADS
PATRON_BIRADS_TYPOS = re.compile(
    r'\bbi\s*[-*.]?\s*rad[ai]?[s]?'      # BI-RADS, BIRADAS, BI-RAD (incluye typo RADAS)
    r'\s*®?\s*:?\s*\(?\s*'
    r'([Ool0-6])'                        # letra O (mayuscula/minuscula), l minuscula, o digito 0-6
    r'\s*([abc])?'
    r'\s*\)?',
    re.IGNORECASE
)

# Tablas de conversion
ROMANOS_A_NUMERO = {"I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6}
PALABRAS_A_NUMERO = {"cero": 0, "uno": 1, "dos": 2, "tres": 3,
                     "cuatro": 4, "cinco": 5, "seis": 6}
TYPOS_A_NUMERO = {"O": 0, "o": 0, "l": 1}

print("Patrones BI-RADS compilados (principal + tolerante a typos)")


Patrones BI-RADS compilados (principal + tolerante a typos)


---

## Paso 3 — Funciones auxiliares

### 3.1 Conversión de numeraciones a entero

Esta función convierte cualquier representación de número (arábigo, romano, palabra) al entero 0-6 que el resto del sistema espera.


In [4]:
def _convertir_a_entero(s):
    """Convierte un string que representa un numero BI-RADS al entero 0-6.
    
    Maneja:
    - Arabigos directos: '0' a '6'
    - Rangos: '3-4', '4-5' (devuelve el mayor, criterio conservador)
    - Romanos: 'I' a 'VI' (insensible a mayusculas)
    - Palabras: 'cero', 'uno', 'dos', etc.
    - Typos: 'O' o 'o' como 0, 'l' como 1
    
    Devuelve None si no se puede convertir.
    """
    if not s:
        return None
    s = s.strip()
    
    # Caso 1: rango como '3-4' -> tomar el mayor
    if "-" in s:
        partes = [p.strip() for p in s.split("-")]
        try:
            numeros = [int(p) for p in partes if p.isdigit()]
            if numeros:
                return max(numeros)
        except ValueError:
            pass
    
    # Caso 2: arabigo simple
    if s.isdigit():
        n = int(s)
        if 0 <= n <= 6:
            return n
    
    # Caso 3: romano
    s_upper = s.upper()
    if s_upper in ROMANOS_A_NUMERO:
        return ROMANOS_A_NUMERO[s_upper]
    
    # Caso 4: palabra
    s_lower = s.lower()
    if s_lower in PALABRAS_A_NUMERO:
        return PALABRAS_A_NUMERO[s_lower]
    
    # Caso 5: typo
    if s in TYPOS_A_NUMERO:
        return TYPOS_A_NUMERO[s]
    
    return None


# Test rapido
casos_test = ["0", "4", "6", "3-4", "4-5", "I", "II", "VI", "cero", "uno", "seis", "O", "o", "l", "X", ""]
print("Conversion de numeraciones a entero:")
for c in casos_test:
    r = _convertir_a_entero(c)
    print(f"  {repr(c):10s} -> {r}")


Conversion de numeraciones a entero:
  '0'        -> 0
  '4'        -> 4
  '6'        -> 6
  '3-4'      -> 4
  '4-5'      -> 5
  'I'        -> 1
  'II'       -> 2
  'VI'       -> 6
  'cero'     -> 0
  'uno'      -> 1
  'seis'     -> 6
  'O'        -> 0
  'o'        -> 0
  'l'        -> 1
  'X'        -> None
  ''         -> None


### 3.2 Localización del bloque CONCLUSIÓN

Esta función busca dónde empieza el bloque de conclusión y dónde termina (al llegar a las recomendaciones o al final del texto).


In [5]:
def _localizar_bloque_conclusion(texto):
    """Localiza el bloque de CONCLUSION en el informe.
    
    Devuelve un dict con: texto, inicio, fin, encabezado.
    Devuelve None si no encuentra ningun encabezado de conclusion.
    """
    if not isinstance(texto, str) or not texto.strip():
        return None
    
    match_conclusion = PATRON_ENCABEZADO_CONCLUSION.search(texto)
    if not match_conclusion:
        return None
    
    inicio_bloque = match_conclusion.end()
    encabezado_encontrado = match_conclusion.group(0).strip().rstrip(":").strip()
    
    # Buscar donde termina el bloque: en el inicio de RECOMENDACIONES o al final
    resto_texto = texto[inicio_bloque:]
    match_recom = PATRON_INICIO_RECOMENDACIONES.search(resto_texto)
    
    if match_recom:
        fin_bloque = inicio_bloque + match_recom.start()
    else:
        fin_bloque = len(texto)
    
    return {
        "texto": texto[inicio_bloque:fin_bloque].strip(),
        "inicio": inicio_bloque,
        "fin": fin_bloque,
        "encabezado": encabezado_encontrado,
    }


# Test rapido sobre 3 informes del corpus
print("Test sobre 3 informes del corpus:")
for i, (idx, row) in enumerate(df.sample(3, random_state=42).iterrows(), 1):
    resultado = _localizar_bloque_conclusion(row["Full_Report"])
    print(f"\nInforme {i} (idx {idx}, BI-RADS {row['BI-RADS']}):")
    if resultado:
        print(f"  Encabezado: '{resultado['encabezado']}'")
        print(f"  Bloque (primeros 150 chars): {resultado['texto'][:150]}...")
    else:
        print("  No se localizo bloque de conclusion")


Test sobre 3 informes del corpus:

Informe 1 (idx 1721, BI-RADS 0):
  Encabezado: 'impresion'
  Bloque (primeros 150 chars): an algunas opacidades nodulares ovaladas, isodensas y de márgenes oscurecidos, que miden entre 3 y 6 mm de diámetro mayor.
No se observan calcificacio...

Informe 2 (idx 1393, BI-RADS 2):
  Encabezado: 'CONCLUSIÓN'
  Bloque (primeros 150 chars): - IMAGEN NODULAR CON CARÁCTER DE GLI EN LA MAMA DERECHA.
- CALCIFICACIONES CON CARACTERES BENIGNOS EN AMBAS MAMAS.
- BI-RADS 2 (Según la ACR)....

Informe 3 (idx 1609, BI-RADS 4):
  Encabezado: 'CONCLUSIÓN'
  Bloque (primeros 150 chars): - Imagen sugerente de proceso inflamatorio en mama derecha.
- Calcificaciones con caracteres benignos en ambas mamas.
Última ecografía mamaria con la ...


---

## Paso 4 — Función pública principal `extraer_birads()`

Esta es la función que cualquier otro módulo del sistema va a importar. Devuelve un diccionario estructurado con toda la información que el resto del pipeline necesita.


In [6]:
def extraer_birads(texto):
    """Extrae el BI-RADS declarado en la conclusion del informe.
    
    Estrategia (en orden):
    1. Localizar el bloque CONCLUSION
    2. Buscar BI-RADS dentro del bloque con patron principal -> confianza alta
    3. Si no, buscar con patron tolerante (typos) -> confianza media
    4. Si no, buscar en todo el texto -> confianza baja
    5. Si no, devolver None -> confianza no_detectado
    
    Returns:
        dict con birads_conclusion, subcategoria, categoria_completa,
        confianza, fuente, encabezado_conclusion, menciones_adicionales,
        rango_detectado, formato_original, error.
    """
    resultado = {
        "birads_conclusion": None,
        "subcategoria": None,
        "categoria_completa": "no detectado",
        "confianza": "no_detectado",
        "fuente": None,
        "encabezado_conclusion": None,
        "menciones_adicionales": [],
        "rango_detectado": False,
        "formato_original": None,
        "error": None,
    }
    
    if not isinstance(texto, str) or not texto.strip():
        resultado["error"] = "Texto vacio o no es string"
        return resultado
    
    # PASO 1: Localizar el bloque conclusion
    bloque = _localizar_bloque_conclusion(texto)
    
    if bloque:
        resultado["encabezado_conclusion"] = bloque["encabezado"]
        
        # PASO 2: buscar BI-RADS dentro del bloque con regex PRINCIPAL
        match = PATRON_BIRADS_PRINCIPAL.search(bloque["texto"])
        if match:
            numero_str = match.group(1)
            subcat = match.group(2)
            numero = _convertir_a_entero(numero_str)
            
            if numero is not None:
                resultado["birads_conclusion"] = numero
                resultado["subcategoria"] = subcat.lower() if subcat else None
                resultado["categoria_completa"] = (
                    f"{numero}{subcat.lower()}" if subcat else str(numero)
                )
                resultado["confianza"] = "alta"
                resultado["fuente"] = "bloque_conclusion_estricto"
                resultado["rango_detectado"] = "-" in numero_str
                
                # Detectar formato original
                if numero_str.upper() in ROMANOS_A_NUMERO:
                    resultado["formato_original"] = "romano"
                elif numero_str.lower() in PALABRAS_A_NUMERO:
                    resultado["formato_original"] = "palabra"
                else:
                    resultado["formato_original"] = "arabigo"
                
                # Si fue un rango, bajar confianza a media
                if resultado["rango_detectado"]:
                    resultado["confianza"] = "media"
        
        # PASO 3: si no se encontro, intentar con typos
        if resultado["birads_conclusion"] is None:
            match_typo = PATRON_BIRADS_TYPOS.search(bloque["texto"])
            if match_typo:
                numero_str = match_typo.group(1)
                subcat = match_typo.group(2)
                numero = _convertir_a_entero(numero_str)
                
                if numero is not None:
                    resultado["birads_conclusion"] = numero
                    resultado["subcategoria"] = subcat.lower() if subcat else None
                    resultado["categoria_completa"] = (
                        f"{numero}{subcat.lower()}" if subcat else str(numero)
                    )
                    resultado["confianza"] = "media"
                    resultado["fuente"] = "bloque_conclusion_tolerante_typos"
                    resultado["formato_original"] = "typo"
    
    # PASO 4: fallback al texto completo
    if resultado["birads_conclusion"] is None:
        match = PATRON_BIRADS_PRINCIPAL.search(texto)
        if not match:
            match = PATRON_BIRADS_TYPOS.search(texto)
        
        if match:
            numero_str = match.group(1)
            subcat = match.group(2)
            numero = _convertir_a_entero(numero_str)
            
            if numero is not None:
                resultado["birads_conclusion"] = numero
                resultado["subcategoria"] = subcat.lower() if subcat else None
                resultado["categoria_completa"] = (
                    f"{numero}{subcat.lower()}" if subcat else str(numero)
                )
                resultado["confianza"] = "baja"
                resultado["fuente"] = "fallback_texto_completo"
                resultado["formato_original"] = "arabigo"
    
    # PASO 5: menciones adicionales (fuera del bloque conclusion)
    if bloque:
        texto_fuera_bloque = texto[:bloque["inicio"]] + texto[bloque["fin"]:]
        for m in PATRON_BIRADS_PRINCIPAL.finditer(texto_fuera_bloque):
            numero_str = m.group(1)
            numero = _convertir_a_entero(numero_str)
            if numero is not None and numero != resultado["birads_conclusion"]:
                if numero not in resultado["menciones_adicionales"]:
                    resultado["menciones_adicionales"].append(numero)
    
    return resultado


print("Funcion extraer_birads() lista")


Funcion extraer_birads() lista


---

## Paso 5 — Tests inline con casos representativos

Antes de aplicar el extractor al corpus completo, lo pongo a prueba con 12 casos sintéticos que cubren distintos escenarios. Si todos pasan, el extractor está listo para validación masiva.


In [7]:
CASOS_TEST = [
    {
        "nombre": "Caso 1: BI-RADS 2 estandar",
        "texto": "MAMOGRAFIA... CONCLUSION: - BI-RADS 2 (Segun la ACR). RECOMENDACIONES: control anual.",
        "esperado_birads": 2,
        "esperado_confianza": "alta",
    },
    {
        "nombre": "Caso 2: BI-RADS (R) 0",
        "texto": "CONCLUSION: BI-RADS \u00ae 0 (Segun la ACR). Requerira ecografia. RECOMENDACIONES: ...",
        "esperado_birads": 0,
        "esperado_confianza": "alta",
    },
    {
        "nombre": "Caso 3: Subcategoria 4A",
        "texto": "CONCLUSION: - BI-RADS 4A (Segun la ACR). RECOMENDACIONES: biopsia.",
        "esperado_birads": 4,
        "esperado_subcategoria": "a",
        "esperado_confianza": "alta",
    },
    {
        "nombre": "Caso 4: VALORACION como encabezado",
        "texto": "MAMOGRAFIA... VALORACION: - BI-RADS \u00ae 3 (segun la ACR). RECOMENDACIONES: control 6 meses.",
        "esperado_birads": 3,
        "esperado_confianza": "alta",
    },
    {
        "nombre": "Caso 5: typo letra O por cero",
        "texto": "CONCLUSION: - BI-RADS O (Segun la ACR). RECOMENDACIONES: ecografia complementaria.",
        "esperado_birads": 0,
        "esperado_confianza": "media",
    },
    {
        "nombre": "Caso 6: typo BI-RADAS",
        "texto": "CONCLUSION: BI-RADAS 4 (Segun la ACR). RECOMENDACIONES: biopsia.",
        "esperado_birads": 4,
        "esperado_confianza": "media",
    },
    {
        "nombre": "Caso 7: rango BI-RADS 4-5 (conservador, toma 5)",
        "texto": "CONCLUSION: BI-RADS 4-5 lesion sospechosa. RECOMENDACIONES: biopsia inmediata.",
        "esperado_birads": 5,
        "esperado_confianza": "media",
        "esperado_rango": True,
    },
    {
        "nombre": "Caso 8: Romano IV (latente)",
        "texto": "CONCLUSION: BI-RADS IV (segun ACR). RECOMENDACIONES: biopsia.",
        "esperado_birads": 4,
        "esperado_confianza": "alta",
    },
    {
        "nombre": "Caso 9: Palabra 'tres'",
        "texto": "CONCLUSION: BI-RADS tres. RECOMENDACIONES: control semestral.",
        "esperado_birads": 3,
        "esperado_confianza": "alta",
    },
    {
        "nombre": "Caso 10: Multiples menciones (ecografia citada antes)",
        "texto": "ECOGRAFIA INFORMA BI-RADS 1. CONCLUSION: BI-RADS 2 (segun ACR). RECOMENDACIONES: control anual.",
        "esperado_birads": 2,
        "esperado_menciones_adicionales": [1],
    },
    {
        "nombre": "Caso 11: Sin bloque CONCLUSION (fallback)",
        "texto": "Informe sin estructura clara, solo dice BI-RADS 2 en alguna parte.",
        "esperado_birads": 2,
        "esperado_confianza": "baja",
    },
    {
        "nombre": "Caso 12: Texto sin BI-RADS",
        "texto": "MAMOGRAFIA NORMAL. CONCLUSION: estudio sin hallazgos. RECOMENDACIONES: control anual.",
        "esperado_birads": None,
        "esperado_confianza": "no_detectado",
    },
]

print("=" * 70)
print("EJECUCION DE TESTS INLINE")
print("=" * 70)

n_pasados = 0
for caso in CASOS_TEST:
    resultado = extraer_birads(caso["texto"])
    
    ok_birads = resultado["birads_conclusion"] == caso["esperado_birads"]
    ok_confianza = ("esperado_confianza" not in caso or
                    resultado["confianza"] == caso["esperado_confianza"])
    ok_subcat = ("esperado_subcategoria" not in caso or
                 resultado["subcategoria"] == caso["esperado_subcategoria"])
    ok_rango = ("esperado_rango" not in caso or
                resultado["rango_detectado"] == caso["esperado_rango"])
    ok_menciones = ("esperado_menciones_adicionales" not in caso or
                    resultado["menciones_adicionales"] == caso["esperado_menciones_adicionales"])
    
    paso = ok_birads and ok_confianza and ok_subcat and ok_rango and ok_menciones
    estado = "PASA" if paso else "FALLA"
    
    if paso:
        n_pasados += 1
        print(f"  [{estado}] {caso['nombre']}")
    else:
        print(f"  [{estado}] {caso['nombre']}")
        print(f"         birads: esperado={caso['esperado_birads']}, obtenido={resultado['birads_conclusion']}")
        print(f"         confianza: esperado={caso.get('esperado_confianza', 'N/A')}, obtenido={resultado['confianza']}")
        if "esperado_subcategoria" in caso:
            print(f"         subcat: esperado={caso['esperado_subcategoria']}, obtenido={resultado['subcategoria']}")
        if "esperado_menciones_adicionales" in caso:
            print(f"         menciones: esperado={caso['esperado_menciones_adicionales']}, obtenido={resultado['menciones_adicionales']}")

print(f"\nResumen: {n_pasados}/{len(CASOS_TEST)} casos pasados")


EJECUCION DE TESTS INLINE
  [PASA] Caso 1: BI-RADS 2 estandar
  [PASA] Caso 2: BI-RADS (R) 0
  [PASA] Caso 3: Subcategoria 4A
  [PASA] Caso 4: VALORACION como encabezado
  [PASA] Caso 5: typo letra O por cero
  [PASA] Caso 6: typo BI-RADAS
  [PASA] Caso 7: rango BI-RADS 4-5 (conservador, toma 5)
  [PASA] Caso 8: Romano IV (latente)
  [PASA] Caso 9: Palabra 'tres'
  [PASA] Caso 10: Multiples menciones (ecografia citada antes)
  [PASA] Caso 11: Sin bloque CONCLUSION (fallback)
  [PASA] Caso 12: Texto sin BI-RADS

Resumen: 12/12 casos pasados


---

## Paso 6 — Validación sobre el corpus completo

Ahora aplico el extractor a los 4 357 informes y comparo el resultado contra la etiqueta del dataset (`BI-RADS`). Esto me dice qué tan bien funciona el extractor en datos reales.


In [8]:
print("Procesando 4 357 informes...")
resultados_corpus = df["Full_Report"].apply(extraer_birads)

df_validacion = df[["BI-RADS", "Full_Report"]].copy()
df_validacion["birads_extraido"] = resultados_corpus.apply(lambda r: r["birads_conclusion"])
df_validacion["confianza"] = resultados_corpus.apply(lambda r: r["confianza"])
df_validacion["fuente"] = resultados_corpus.apply(lambda r: r["fuente"])
df_validacion["subcategoria"] = resultados_corpus.apply(lambda r: r["subcategoria"])
df_validacion["rango_detectado"] = resultados_corpus.apply(lambda r: r["rango_detectado"])
df_validacion["formato_original"] = resultados_corpus.apply(lambda r: r["formato_original"])
df_validacion["menciones_adicionales"] = resultados_corpus.apply(lambda r: r["menciones_adicionales"])
df_validacion["encabezado_conclusion"] = resultados_corpus.apply(lambda r: r["encabezado_conclusion"])

print("Procesamiento completado\n")
print(df_validacion[["BI-RADS", "birads_extraido", "confianza", "fuente"]].head(10))


Procesando 4 357 informes...
Procesamiento completado

   BI-RADS  birads_extraido confianza                      fuente
0        0                0      alta  bloque_conclusion_estricto
1        2                2      alta  bloque_conclusion_estricto
2        0                0      alta  bloque_conclusion_estricto
3        1                1      alta  bloque_conclusion_estricto
4        0                0      alta  bloque_conclusion_estricto
5        0                0      alta  bloque_conclusion_estricto
6        2                2      alta  bloque_conclusion_estricto
7        2                2      alta  bloque_conclusion_estricto
8        2                2      alta  bloque_conclusion_estricto
9        2                2      alta  bloque_conclusion_estricto


### 6.1 Tasa de coincidencia contra la etiqueta del dataset


In [9]:
df_validacion["coincide"] = (
    df_validacion["birads_extraido"] == df_validacion["BI-RADS"]
)

n_total = len(df_validacion)
n_extraido = df_validacion["birads_extraido"].notna().sum()
n_no_extraido = n_total - n_extraido
n_coincide = df_validacion["coincide"].sum()
n_divergente = n_extraido - n_coincide

print("=" * 70)
print("RESULTADOS DE EXTRACCION")
print("=" * 70)
print(f"Total informes:                  {n_total}")
print(f"  Con BI-RADS extraido:          {n_extraido} ({100*n_extraido/n_total:.2f}%)")
print(f"  Sin BI-RADS extraido:          {n_no_extraido} ({100*n_no_extraido/n_total:.2f}%)")
print()
print(f"  Coincide con etiqueta dataset: {n_coincide} ({100*n_coincide/n_total:.2f}%)")
print(f"  Diverge de etiqueta dataset:   {n_divergente} ({100*n_divergente/n_total:.2f}%)")

print("\n" + "=" * 70)
print("DISTRIBUCION POR NIVEL DE CONFIANZA")
print("=" * 70)
print(df_validacion["confianza"].value_counts())

print("\n" + "=" * 70)
print("DISTRIBUCION POR FUENTE DE EXTRACCION")
print("=" * 70)
print(df_validacion["fuente"].value_counts(dropna=False))


RESULTADOS DE EXTRACCION
Total informes:                  4357
  Con BI-RADS extraido:          4357 (100.00%)
  Sin BI-RADS extraido:          0 (0.00%)

  Coincide con etiqueta dataset: 4354 (99.93%)
  Diverge de etiqueta dataset:   3 (0.07%)

DISTRIBUCION POR NIVEL DE CONFIANZA
confianza
alta     4225
baja      130
media       2
Name: count, dtype: int64

DISTRIBUCION POR FUENTE DE EXTRACCION
fuente
bloque_conclusion_estricto           4225
fallback_texto_completo               130
bloque_conclusion_tolerante_typos       2
Name: count, dtype: int64


### 6.2 Matriz de coincidencia BI-RADS extraído vs BI-RADS etiquetado


In [10]:
matriz = pd.crosstab(
    df_validacion["BI-RADS"],
    df_validacion["birads_extraido"],
    dropna=False,
    margins=True,
    margins_name="Total"
)
print("MATRIZ: filas = BI-RADS etiqueta dataset, columnas = BI-RADS extraido")
print(matriz)


MATRIZ: filas = BI-RADS etiqueta dataset, columnas = BI-RADS extraido
birads_extraido    0    1     2   3   4   5  6  Total
BI-RADS                                              
0                966    0     0   0   0   0  0    966
1                  1  593     2   0   0   0  0    596
2                  0    0  2635   0   0   0  0   2635
3                  0    0     0  87   0   0  0     87
4                  0    0     0   0  52   0  0     52
5                  0    0     0   0   0  16  0     16
6                  0    0     0   0   0   0  5      5
Total            967  593  2637  87  52  16  5   4357


---

## Paso 7 — Análisis de divergencias

Reviso los casos donde el extractor difiere de la etiqueta del dataset. Estos son los más interesantes porque revelan inconsistencias reales del informe (BI-RADS en conclusión distinto del considerado por el dataset), bugs del extractor o casos especiales (ecografía citada, múltiples lesiones).


In [11]:
divergentes = df_validacion[
    (df_validacion["birads_extraido"].notna()) &
    (~df_validacion["coincide"])
].copy()

print(f"Total casos divergentes: {len(divergentes)}")
print()
print("Distribucion de divergencias:")
distr_div = pd.crosstab(divergentes["BI-RADS"], divergentes["birads_extraido"])
print(distr_div)

print("\n" + "=" * 70)
print("EJEMPLOS DE DIVERGENCIA (hasta 5)")
print("=" * 70)
for i, (idx, row) in enumerate(divergentes.head(5).iterrows(), 1):
    print(f"\n--- Divergencia {i} (idx {idx}) ---")
    print(f"  Etiqueta dataset:    BI-RADS {row['BI-RADS']}")
    print(f"  Extraido (conclus.): BI-RADS {row['birads_extraido']}")
    print(f"  Confianza:           {row['confianza']}")
    print(f"  Fuente:              {row['fuente']}")
    print(f"  Menciones adic.:     {row['menciones_adicionales']}")
    print(f"  Final del Full_Report:")
    print(f"    ...{row['Full_Report'][-250:]}")


Total casos divergentes: 3

Distribucion de divergencias:
birads_extraido  0  2
BI-RADS              
1                1  2

EJEMPLOS DE DIVERGENCIA (hasta 5)

--- Divergencia 1 (idx 1079) ---
  Etiqueta dataset:    BI-RADS 1
  Extraido (conclus.): BI-RADS 0
  Confianza:           alta
  Fuente:              bloque_conclusion_estricto
  Menciones adic.:     [1]
  Final del Full_Report:
    ... A DETERMINAR.
-   CALCIFICACIONES CON CARACTERES BENIGNOS EN AMBAS MAMAS.
-   BI-RADS 0 (Según la ACR). Requerirá de estudio complementario.
RECOMENDACIONES:
- SE SUGIERE CORRELACIÓN CON ECOGRAFÍA MAMARIA ACTUALIZADA PARA POSTERIOR RECATEGORIZACIÓN.

--- Divergencia 2 (idx 1111) ---
  Etiqueta dataset:    BI-RADS 1
  Extraido (conclus.): BI-RADS 2
  Confianza:           alta
  Fuente:              bloque_conclusion_estricto
  Menciones adic.:     [1]
  Final del Full_Report:
    ...FÍA ANTERIOR REALIZADA EL 13-09-18.
ECOGRAFÍA MAMARIA PRACTICADA EL 20-09-18 INFORMA BI-RADS 1.
CONCLUSIÓN:
- CALCIF

### 7.1 Casos no extraídos


In [12]:
no_extraidos = df_validacion[df_validacion["birads_extraido"].isna()]
print(f"Casos sin extraccion: {len(no_extraidos)}")

if len(no_extraidos) > 0:
    print("\nEjemplos:")
    for idx, row in no_extraidos.head(5).iterrows():
        print(f"\n--- idx {idx} | etiqueta dataset BI-RADS {row['BI-RADS']} ---")
        print(f"Full_Report (primeros 400 chars):")
        print(row["Full_Report"][:400])


Casos sin extraccion: 0


### 7.2 Subcategorías detectadas


In [13]:
con_subcat = df_validacion[df_validacion["subcategoria"].notna()]
print(f"Informes con subcategoria detectada: {len(con_subcat)}")
print("\nDistribucion de subcategorias:")
print(con_subcat.groupby(["birads_extraido", "subcategoria"]).size())


Informes con subcategoria detectada: 32

Distribucion de subcategorias:
birads_extraido  subcategoria
4                a               14
                 b                6
                 c               12
dtype: int64


### 7.3 Menciones múltiples


In [14]:
con_menciones_adicionales = df_validacion[
    df_validacion["menciones_adicionales"].apply(lambda x: len(x) > 0)
]
print(f"Informes con menciones BI-RADS adicionales fuera de la conclusion: {len(con_menciones_adicionales)}")

if len(con_menciones_adicionales) > 0:
    print("\nEjemplos:")
    for idx, row in con_menciones_adicionales.head(3).iterrows():
        print(f"\n--- idx {idx} ---")
        print(f"  BI-RADS etiqueta:      {row['BI-RADS']}")
        print(f"  BI-RADS conclusion:    {row['birads_extraido']}")
        print(f"  Menciones adicionales: {row['menciones_adicionales']}")


Informes con menciones BI-RADS adicionales fuera de la conclusion: 2

Ejemplos:

--- idx 1079 ---
  BI-RADS etiqueta:      1
  BI-RADS conclusion:    0
  Menciones adicionales: [1]

--- idx 1111 ---
  BI-RADS etiqueta:      1
  BI-RADS conclusion:    2
  Menciones adicionales: [1]


---

## Paso 8 — Guardar resultados y conclusiones


In [15]:
import os
import json

os.makedirs("./anexos", exist_ok=True)
df_validacion.to_csv("./anexos/validacion_extractor_birads.csv", index=False)
print("OK Guardado: notebooks/anexos/validacion_extractor_birads.csv")

resumen = {
    "total_informes": int(n_total),
    "extraidos": int(n_extraido),
    "no_extraidos": int(n_no_extraido),
    "tasa_extraccion": round(100 * n_extraido / n_total, 2),
    "coincide_etiqueta": int(n_coincide),
    "diverge_etiqueta": int(n_divergente),
    "tasa_coincidencia": round(100 * n_coincide / n_total, 2),
    "por_confianza": df_validacion["confianza"].value_counts().to_dict(),
}

with open("./anexos/resumen_extractor_birads.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

print("OK Guardado: notebooks/anexos/resumen_extractor_birads.json")
print("\nResumen:")
for k, v in resumen.items():
    print(f"  {k}: {v}")


OK Guardado: notebooks/anexos/validacion_extractor_birads.csv
OK Guardado: notebooks/anexos/resumen_extractor_birads.json

Resumen:
  total_informes: 4357
  extraidos: 4357
  no_extraidos: 0
  tasa_extraccion: 100.0
  coincide_etiqueta: 4354
  diverge_etiqueta: 3
  tasa_coincidencia: 99.93
  por_confianza: {'alta': 4225, 'baja': 130, 'media': 2}


---

## Conclusiones (a completar después de ejecutar)

### Métricas del extractor
- Tasa de extracción: _____ %
- Tasa de coincidencia con etiqueta dataset: _____ %
- Casos sin extracción: _____ (motivo principal: _____)
- Casos divergentes: _____ (interpretación: _____)

### Hallazgos relevantes
- Casos que usaron el fallback de typos: _____
- Subcategorías 4a/4b/4c detectadas: _____
- Informes con menciones múltiples: _____

### Decisión para el MVP
Si la tasa de extracción es >99% y la tasa de coincidencia es alta, el extractor está listo. Si la tasa de coincidencia es <95%, conviene revisar la lógica de selección entre múltiples menciones.

### Siguiente paso
Una vez validado el extractor, pasar a `extractor_recomendacion.py` (más simple: usa principalmente la columna `Recommendations` y solo recurre a parsing si está vacía).
